In [1]:
! pip install datasets transformers[torch] 'accelerate>=0.26.0' peft bitsandbytes

In [2]:
distilled_model = "../CustomProtocol/LA-Framework-dist-peft/results/distilled-Qwen3-1.7B/bs16-lr1e-05-G4-N1-NN1-lm1-len384-lora-8-32-0.1/pe2_rs0.5_nr128_ln_sr_tm0.2/1773"
distilled_model = "LA-framework-dist/results/distilled-Qwen3-1.7B/bs8-lr1e-05-G4-N1-NN1-lm1-len512-lora-8-32-0.1/pe2_rs0.5_nr64_ln_sr_tm0.2/2500"


In [6]:
from datasets import load_dataset

def load_io_dataset(split, uf_n=90):
    data_files = []
    data_files.append(f"TTE_uf{uf_n}/TTE_with_IO_{split}.json")
    dataset = load_dataset("json", data_files=f"TTE_uf{uf_n}/TTE_with_IO_{split}.json")

    def format_example(example):
        return {
            "text": f"Prompt: {example['input']}\nResponse: {example['output']}\n"
        }

    dataset = dataset["train"].map(format_example)
    return dataset

train_dataset = load_io_dataset("train")
test_dataset = load_io_dataset("test")
eval_dataset = load_io_dataset("eval")
print(train_dataset)
print(test_dataset)
print(eval_dataset)


Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 4430
})
Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 4430
})
Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 4430
})


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(distilled_model)
model = AutoModelForCausalLM.from_pretrained(
    distilled_model,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [7]:
prompt = eval_dataset[0]['input']
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=600, min_new_tokens=100)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Based on the user's watching history and the film's rating, order the list of candidate films. Order the films where the first one of the list is the most likely to be watched by the user. The output should only contain the ordered list of the recommended films bounded by the  special strings '%% START RECOMMENDED LIST %%' and '%% END LIST %%'. Here follows the user's history and the list of candidate films.
%% START HISTORY %%
Movie name: Toy Story 2 (1999)	Rating:4
Movie name: Airplane! (1980)	Rating:4
Movie name: Pleasantville (1998)	Rating:3
Movie name: Dumbo (1941)	Rating:5
Movie name: Princess Bride, The (1987)	Rating:3
Movie name: Snow White and the Seven Dwarfs (1937)	Rating:4
Movie name: Miracle on 34th Street (1947)	Rating:4
Movie name: Ponette (1996)	Rating:4
Movie name: Schindler's List (1993)	Rating:5
Movie name: Aladdin (1992)	Rating:4
%% END HISTORY %%
%% START CANDIDATES %%
Movie name: 13th Warrior, The (1999)
Movie name: Dogma (1999)
Movie name: Unforgiven (1992)
Movie

In [9]:
import torch
from tqdm.auto import tqdm   # optional, for nicer progress

inference_model = model

def split_list(text, start_token, end_token):
    if not (start_token + "\n" in text):
            return []
    phrase = text.split(start_token + "\n")
    movie_list = phrase[1].split("\n" + end_token)[0]
    movie_list = movie_list.split("\n")
    for i, movie in enumerate(movie_list):
        movie_list[i] = movie.split("\t")
    return movie_list


def perform_and_return_comparison(sample):
    #prompt = sample['text']
    inputs = tokenizer(sample['input'], return_tensors="pt").to("cuda")
    while True:
        output = inference_model.generate(**inputs, max_new_tokens=600)
        rating = tokenizer.decode(output[0], skip_special_tokens=True)
        if ("%% START RECOMMENDED LIST %%\n" in rating):
            break
    
    rating = split_list(rating, "%% START RECOMMENDED LIST %%", "%% END LIST %%")
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    return {'input': sample['input'],
            'rating': rating}

In [11]:
import torch
from tqdm.auto import tqdm   # optional, for nicer progress
import pandas as pd

# Parallel LLM generation of evaluation's results

def perform_and_return_comparison_batched(batch):
    # batch is a dict with lists: {'input': [...], 'text': [...]}
    inputs = tokenizer(
        batch['input'],
        return_tensors="pt",padding=True,
        truncation=True,
        max_length=2048,  # ← set sensible value, e.g. 2048
        padding_side='left'
    ).to("cuda")

    with torch.inference_mode():           # saves a bit of memory
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=600
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    wo = pd.DataFrame()
    wo['dec']=decoded
    wo.to_json(f"batched_tmp/partial.json")
    return {
        #'results': results,
        'decoded': decoded} 

# This is usually enough parallelism for one GPU
eval_output = eval_dataset.map(
    perform_and_return_comparison_batched,
    batched=True,
    batch_size=64,          # ← tune this: 4–16 most common sweet spot
                           # larger = better GPU util, but risk of OOM
    desc="Evaluating",
    remove_columns=eval_dataset.column_names,  # optional — keep only new fields
)

eval_output.to_json("90_uf90_peft_distilled_eval_processed_parallel.json")

Evaluating:   0%|          | 0/4430 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

31356388

In [12]:
from datasets import load_dataset

def load_io_dataset(split, uf_n=40):
    data_files = []
    data_files.append(f"TTE_uf{uf_n}/TTE_with_IO_{split}.json")
    dataset = load_dataset("json", data_files=f"TTE_uf{uf_n}/TTE_with_IO_{split}.json")

    def format_example(example):
        return {
            "text": f"Prompt: {example['input']}\nResponse: {example['output']}\n"
        }

    dataset = dataset["train"].map(format_example)
    return dataset

train_dataset = load_io_dataset("train")
test_dataset = load_io_dataset("test")
eval_dataset = load_io_dataset("eval")
print(train_dataset)
print(test_dataset)
print(eval_dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4430 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4430 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4430 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 4430
})
Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 4430
})
Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 4430
})


In [13]:
import torch
from tqdm.auto import tqdm   # optional, for nicer progress
import pandas as pd

# Parallel LLM generation of evaluation's results

def perform_and_return_comparison_batched(batch):
    # batch is a dict with lists: {'input': [...], 'text': [...]}
    inputs = tokenizer(
        batch['input'],
        return_tensors="pt",padding=True,
        truncation=True,
        max_length=1024,  # ← set sensible value, e.g. 2048
        padding_side='left'
    ).to("cuda")

    with torch.inference_mode():           # saves a bit of memory
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=600
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    wo = pd.DataFrame()
    wo['dec']=decoded
    wo.to_json(f"batched_tmp/partial.json")
    return {
        #'results': results,
        'decoded': decoded} 

# This is usually enough parallelism for one GPU
eval_output = eval_dataset.map(
    perform_and_return_comparison_batched,
    batched=True,
    batch_size=64,          # ← tune this: 4–16 most common sweet spot
                           # larger = better GPU util, but risk of OOM
    desc="Evaluating",
    remove_columns=eval_dataset.column_names,  # optional — keep only new fields
)

eval_output.to_json("40_ml1024_peft_distilled_eval_processed_parallel.json")

Evaluating:   0%|          | 0/4430 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

21622244